# 17. Order totals that grow after explode

**Track:** pyspark | **Difficulty:** medium | **Tags:** explode, arrays, nested data, grain, aggregation

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("017-explode-double-count")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "orders": """
UEFSMRUEFeAOFcoHTBXsARUAEgAAsAcIIU4ABQEAIg0IACMNCAAkDQgAJQ0IACYNCAAnDQgAKA0IACkNCAAqDQgAKw0IACwNCAAt
DQgALg0IAC8NCAAwDQgAMQ0IADINCAAzDQgANA0IADUNCAA2DQgANw0IADgNCAA5DQgAOg0IADsNCAA8DQgAPQ0IAD4NCAA/DQgA
QA0IAEENCABCDQgAQw0IAEQNCABFDQgARg0IAEcNCABIDQgASQ0IAEoNCABLDQgATA0IAE0NCABODQgATw0IAFANCABRDQgAUg0I
AFMNCABUDQgAVQ0IAFYNCABXDQgAWA0IAFkNCABaDQgAWw0IAFwNCABdDQgAXg0IAF8NCABgDQgAYQ0IAGINCABjDQgAZA0IAGUN
CABmDQgAZw0IAGgNCABpDQgAag0IAGsNCABsDQgAbQ0IAG4NCABvDQgAcA0IAHENCAByDQgAcw0IAHQNCAB1DQgAdg0IAHcNCAB4
DQgAeQ0IAHoNCAB7DQgAfA0IAH0NCAB+DQgAfw0IAIANCACBDQgAgg0IAIMNCACEDQgAhQ0IAIYNCACHDQgAiA0IAIkNCACKDQgA
iw0IAIwNCACNDQgAjg0IAI8NCACQDQgAkQ0IAJINCACTDQgAlA0IPJVOAAAAAAAAlk4AAAAAAAAVABXkARXqASwV7AEVEBUGFQYc
GAiWTgAAAAAAABgIIU4AAAAAAAAWACgIlk4AAAAAAAAYCCFOAAAAAAAAEREAAABy8HEDAAAA7AEBBx+AgGBAKBgOiIRiwWg4HpCI
ZEKpWC6YjGbD6Xg+oJBoRCqZTqiUasVquV6wmGxGq9luuJxux+v5fsCgcEgsGo/IpHLJbDqf0Kh0Sq1ar9isdsvter/gsHhMLpvP
6LR6zW673/C4fE6vAwAVBBWQAxXiAUwVMhUAEgAAyAEIPAEABQEARQ0IADoNCABBDQgAMQ0IADANCABDDQgAMg0IAD8NCAA1DQgA
OA0IAC4NCAAtDQgANw0IADMNCAA7DQgAOQ0IAEANCABCDQgANA0IAD4NCAA2DQgAPQ0IPEQBAAAAAAAALwEAAAAAAAAVABWoARWu
ASwV7AEVEBUGFQYcGAhFAQAAAAAAABgILQEAAAAAAAAWACgIRQEAAAAAAAAYCC0BAAAAAAAAEREAAABU8FMDAAAA7AEBBR8giEFK
MQcdko5aKzHTFHAPRkncpBUq9+xlws7DQJWxmQMSmAQQ6hpaQj3GiqnOLDMjU2chkF5EIyksHBsriWQQPbEshUcy0UklAwAVBBUo
FSxMFQgVABIAABRMAAAAOjQAAAATJAAAAAAAAAAAJqwVABVOFVIsFewBFRAVBhUGHBgFAAAAOjQYBQAAAAAAFgAoBQAAADo0GAUA
AAAAABERAAAAJ5gDAAAA7AEBAh+EfvYpLXFaltl+oaJe+onbWGvlXOX7HWhWZdfVagcVBBXwBhXcAkwVUBUAEgAAuAMoBwAAAFNL
VS0wMTcZCwA1FQsEMDgVCwQxMxULBDI0FQsAMxkLBDExGRYAORULBDIwFQsAMRkLADMZWAAwHTcZYwAwGY8AMhkWADMdQgAyGVgZ
bgAxGY8AMBkLBDE2FSwAMh15GecAMhkhADEZIQAyGVgAMxkhADAZeQAyHUIZsAAzHQsZIQAzGecAMBnRADAZWAAxGVgANBnnADAZ
hDAyMgcAAABTS1UtMDIxFQAV4AQV2gQsFb4EFRAVBhUGHDYSKAdTS1UtMDQwGAdTS1UtMDAxEREAAACwAvA8JQAAAEmuszle83i4
e5LupqGKdwM73uY6ns52mszd3bvp0cWTtqcb3jsvAAAAUAQFIEmSIUmSFAQDIUmSIh0G8OceBAUhSZIkQZK0AQQDIEmSQgQCAAZH
QCAMRGEcApIowcI0CBEczvBAyRBJE0MRlXVRWGZFmoME24AJHNYBzRAFgkcpx7NRmRFYn5FpVSY8CAdd4KUQkdF12uCFQSRWI5lt
YhQ8mcRASjIUQ4kZjUdOTeOBmTlOTeVUwpeRTPWIEnQ9QFKNBcY9H7QJBDckGxYdn2FlUXRWZdgZ1XJNRjAqFSRSglYdjDBCkjNJ
mFZuYuB5XAB0ZgFoi6AgSoUoUBKEW3dekVY63zN9UMSUoIY9hOEoidFkxUMZ4yI0ovlx53cEEOYQVQIAFQQVIBUkTBUIFQASAAAQ
PAMAAAACAAAABAAAAAEAAAAVABXIAhXCAiwVvgQVEBUGFQYcGAQEAAAAGAQBAAAAFgAoBAQAAAAYBAEAAAAREQAAAKQB8DwlAAAA
Sa6zOV7zeLh7ku6moYp3Azve5jqeznaazN3du+nRxZO2pxveOy8AAABQBAUgSZIhSZIUBAMhSZIiHQbwWx4EBSFJkiRBkrQBBAMg
SZJCBAIAAkeU9OgfSR+zIUycyZg/0A/YfHyL96+sishjgE2Q6tYh7nphAoUq/fK6ByAbA6E2bGoLjNecuAOdrwibMXZYFtd+aWw+
O5YFFQQVqhUVshVMFaIEFQASAADVCpgAAADlsAAAAVjsAAAExFAAAAHg3AAABVcwAAABwZwAAAPEdAAABfQBCvQpBb/EAAACoQwA
AABhqAAABJMYAAABlLAAAAXPHAAAARs0AAAE6pgAAAOFkAAABcW8AAAEYwwAAAQTwAAABcv8AAACFNAAAAWdgAAABEpwAAAE0ygA
AACCFAAAA80MAAADSvgAAACBTAAABFT8AAABfUAAAAQLjAAABEwAAAAEpjwAAAUniAAAAdkMAAACJgAAAAEDxAAAAZiYAAAEfNQA
AARt/AAAAjOsAAAFjXwAAAT7yAAAAlhkAAAEZcgAAAGLtAAAAPv0AAADFwQAAATVHAAABhm4AAABjUQAAAFFZAAAADzwAAAB3VgA
AAB9yAAABAVMAAACxWAAAAX9/AAAAW08AAABpFAAAAJAkAAABWVAAAAECGwAAABwHAAAAkNMAAADUAwAAAD6ZAAAANmUAAAGDmQA
AAOPVAAABV3UAAAGCuAAAANHEAAAAuMQAAAD41AAAAESnAAABbrMAAAFA/wAAAOtBAAAAKyoAAAA/3gAAAVcqAAAA/kwAAAEUwgA
AAOe9AAABHVoAAABhwQAAAPVQAAAA3sEAAAESOAAAASLSAAAAJhYAAAAbPwAAAI8RAAAA96gAAADwhwAAAR5UAAAAhCEAAADG7QA
AALPJAAAAlmQAAAFHJgAAARElAAAAeyUAAAAu+QAAAMILAAAAwXUAAAEUXgAAAWNGAAAAHbAAAABBuQAAADKvAAABZn8AAAFhBwA
AAGKiAAABA90AAABzbgAAAEh2AAAATxoAAAFBrgAAAGrWAAABggkAAAA3RgAAAKnsAAAAJ0IAAABGUAAAAK4GAAABYy0AAAEXAQA
AAQynAAABdPMAAAFUPAAAAXB1AAABMyEAAAC9pgAAATZaAAAAbowAAAA39QAAAHOHAAAAbE0AAAE0ZgAAAXDAAAAAjIcAAADDmwA
AAVPxAAAAipMAAAFV5QAAAHkxAAAAmTkAAAF6hAAAACkdAAAAaV8AAABp9QAAACvZAAABUAkAAAA0PwAAAGuFAAAA/hoAAAC/zAA
AASiuAAAAWwQAAADT0QAAAPShAAABCTwAAAEdvgAAAMBiAAABX8IAAABP4gAAAD6AAAAAm3gAAAFxJAAAAF2nAAABf00AAAB4zQA
AAF+0AAAAgRoAAAAKcwAAAHO5AAAAe7sAAADVegAAAMWoAAAArPMAAACKFgAAAWC8AAAA/CYAAAE/7AAAAL3YAAAAgIQAAABvhgA
AASUqAAAACkEAAACRzQAAAWm4AAAA5+8AAADYTwAAALjdAAAAOjQAAAEFnwAAASWOAAAAheMAAAE7hwAAAPAjAAABOtgAAADF2gA
AABsNAAABM4UAAAEjswAAADUHAAAAr/oAAABcYgAAAT0XAAAANjMAAAC3fwAAAL8dAAABKnAAAAAyZAAAAIgiAAAAFZUAAABhEgA
AAOEZAAAAk6gAAADmXwAAAKoeAAAAJCIAAAFocwAAADPbAAAAsdUAAAEYkQAAAP/cAAAArBIAAACDJwAAASFbAAAAH1kAAADY/gA
AAEwsAAAAr8gAAADGiQAAAMBJAAAAc6AAAABeVgAAASbTAAABFbwAAAEiVQAAAP1SAAAA3z4AAABLAAAAAWBYAAAA+CUAAACjkwA
AASqiAAAAraIAAAAnjQAAAUJEAAAAE4gAAAEwMwAAAUKPAAABddQAAAGEegAAAWaYAAABAKQAAABnhAAAAGSWAAAAtzQAAADL6AA
AAKUjAAABFSYAAAErUQAAAVnNAAAA3hIAAADsCQAAAE2jAAAAKTYFQAVsgYVrgYsFb4EFRAVBhUGHBgFAAAGGbgYBQAAACkEFgAo
BQAABhm4GAUAAAApBBERAAAAmQPwPCUAAABJrrM5XvN4uHuS7qahincDO97mOp7OdprM3d276dHFk7anG947LwAAAFAEBSBJkiFJ
khQEAyFJkiIdBvRQAR4EBSFJkiRBkrQBBAMgSZJCBAIACUcAAggYQKCAgQMIEihYwKCBgwcQIkiYQKGChQsYMmjYwKGDhw8gQogY
QaKEiRMoUqhYwaKFixcwYsiYQaOGjRs4cujYwaOHjx8UgAQRMoRIESNHkCRRsoRJEydPoESRMoVKFStXsGTRsoVLFy9fwIQRM4ZM
GTNn0KRRs4ZNGzdv4MSRM4dOHTt38OTRs4dPHz9/AAUSNIhQIUOHECVStIhRI0ePIEWSNIlSJUuXMGXStIlTJ0+fQIUSNYpUKVOn
UKVStYpVK1evYMWSNYtWLVu3cOXStYtXhF6+fgELJmwYsWLGjiFLpmwZs2bOnkGLJm0atWrWrmHLpm0bt27ePn0DF07cOHLlzJ1D
l05dmnXs2rl7By+evHn06tm7hy+fvn38+vn7BzCgwDoDCRY0eBBhQoULGTZ0+BAiAAAVBBmsNQAYBnNjaGVtYRUIABUEJQIYCG9y
ZGVyX2lkABUEJQIYC2N1c3RvbWVyX2lkABUOFQoVAhgMc2hpcHBpbmdfZmVlJQoVBBUULFwVBBUUAAAANQIYBWl0ZW1zFQIVBkw8
AAAANQQYBGxpc3QVAgA1AhgHZWxlbWVudBUGABUMJQIYA3NrdSUATBwAAAAVAiUCGANxdHkAFQ4VChUCGAp1bml0X3ByaWNlJQoV
BBUULFwVBBUUAAAAFuwBGRwZbCYAHBUEGTUABhAZGAhvcmRlcl9pZBUCFuwBFuoRFtoKJvQHJggcGAiWTgAAAAAAABgIIU4AAAAA
AAAWACgIlk4AAAAAAAAYCCFOAAAAAAAAEREAGSwVBBUAFQIAFQAVEBUCADwpBhkmAOwBAAAAJgAcFQQZNQAGEBkYC2N1c3RvbWVy
X2lkFQIW7AEW3AUWtAQm5Awm4gocGAhFAQAAAAAAABgILQEAAAAAAAAWACgIRQEAAAAAAAAYCC0BAAAAAAAAEREAGSwVBBUAFQIA
FQAVEBUCADwpBhkmAOwBAAAAJgAcFQ4ZNQAGEBkYDHNoaXBwaW5nX2ZlZRUCFuwBFvoBFoICJt4PJpYPHBgFAAAAOjQYBQAAAAAA
FgAoBQAAADo0GAUAAAAAABERABksFQQVABUCABUAFRAVAgA8KQYZJgDsAQAAACYAHBUMGTUABhAZSAVpdGVtcwRsaXN0B2VsZW1l
bnQDc2t1FQIWvgQWyAwWrggmlBQmmBEcNhIoB1NLVS0wNDAYB1NLVS0wMDEREQAZLBUEFQAVAgAVABUQFQIAPBa0Hhkm7AHSAhlW
CAoAAKwEAAAAJgAcFQIZNQAGEBlIBWl0ZW1zBGxpc3QHZWxlbWVudANxdHkVAha+BBboAxbmAyaGGibGGRwYBAQAAAAYBAEAAAAW
ACgEBAAAABgEAQAAABERABksFQQVABUCABUAFRAVAgA8KSbsAdICGVYICgAArAQAAAAmABwVDhk1AAYQGUgFaXRlbXMEbGlzdAdl
bGVtZW50CnVuaXRfcHJpY2UVAha+BBbqHBbuHCaAMyasHRwYBQAABhm4GAUAAAApBBYAKAUAAAYZuBgFAAAAKQQREQAZLBUEFQAV
AgAVABUQFQIAPCkm7AHSAhlWCAoAAKwEAAAAFtpGFuwBJggWkjoAGRwYDEFSUk9XOnNjaGVtYRiYBS8vLy8vK2dCQUFBUUFBQUFB
QUFLQUF3QUJnQUZBQWdBQ2dBQUFBQUJCQUFNQUFBQUNBQUlBQUFBQkFBSUFBQUFCQUFBQUFRQUFBQjhBUUFBTkFFQUFQQUFBQUFF
QUFBQXBQNy8vd0FBQVF3VUFBQUFIQUFBQUFRQUFBQUJBQUFBRkFBQUFBVUFBQUJwZEdWdGN3QUFBRWovLy8vUS92Ly9BQUFCRFJ3
QUFBQWtBQUFBQkFBQUFBTUFBQUI4QUFBQVRBQUFBQlFBQUFBRUFBQUFhWFJsYlFBQUFBQjgvLy8vQlAvLy93QUFBUWNRQUFBQUhB
QUFBQVFBQUFBQUFBQUFDZ0FBQUhWdWFYUmZjSEpwWTJVQUFIVC8vLzhLQUFBQUFnQUFBRGovLy84QUFBRUNFQUFBQUJRQUFBQUVB
QUFBQUFBQUFBTUFBQUJ4ZEhrQUlQLy8vd0FBQUFFZ0FBQUFaUC8vL3dBQUFRVVFBQUFBR0FBQUFBUUFBQUFBQUFBQUF3QUFBSE5y
ZFFBRUFBUUFCQUFBQUl6Ly8vOEFBQUVIRUFBQUFDZ0FBQUFFQUFBQUFBQUFBQXdBQUFCemFHbHdjR2x1WjE5bVpXVUFBQUFBQ0FB
TUFBUUFDQUFJQUFBQUNnQUFBQUlBQUFETS8vLy9BQUFCQWhBQUFBQWNBQUFBQkFBQUFBQUFBQUFMQUFBQVkzVnpkRzl0WlhKZmFX
UUF2UC8vL3dBQUFBRkFBQUFBRUFBVUFBZ0FCZ0FIQUF3QUFBQVFBQkFBQUFBQUFBRUNFQUFBQUNRQUFBQUVBQUFBQUFBQUFBZ0FB
QUJ2Y21SbGNsOXBaQUFBQUFBSUFBd0FDQUFIQUFnQUFBQUFBQUFCUUFBQUFBPT0AGCBwYXJxdWV0LWNwcC1hcnJvdyB2ZXJzaW9u
IDI1LjAuMRlsHAAAHAAAHAAAHAAAHAAAHAAAAFwGAABQQVIx
""",
}

_data_dir = pathlib.Path("data") / "017-explode-double-count"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

orders = spark.read.parquet((_data_dir / "orders.parquet").resolve().as_posix())
orders.createOrReplaceTempView("orders")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'customer_id': '6f28a2c0eccf7183086f26c547accc6873afe5e2caffb30dc4176a2a9a69f756',
                         'items_value': '2aa1754310655a2bd5dc02cac1e2c0874082d3496e1720c2c73a8257017ac097',
                         'orders': '3241623440afc1a1b76c9f794ffe86649efa97fc3aff2d51b2988edec3f69a7f',
                         'shipping_total': '1bb738c71772dadc160ba5be8442890e85026bbba8aa7d014df4b42f9006a185',
                         'total_units': 'bce0570d4a00961bedf7b8f1a99153bda542d0e74daa0f4027ce9b685bd260bf'},
 'columns': [['customer_id', 'bigint'],
             ['items_value', 'decimal(14,2)'],
             ['orders', 'bigint'],
             ['shipping_total', 'decimal(12,2)'],
             ['total_units', 'bigint']],
 'fingerprint': 'aec3d4ea6d5fdb164cffd8a1221e9f83dbefdf9c81a2695d553215fc2c5a120e',
 'order_matters': False,
 'row_count': 25}

check = make_check(EXPECTED)

## The situation

Orders arrive as JSON and are stored with the line items nested inside each order.
The finance team needs one summary row per customer that mixes two kinds of numbers:
values that belong to the order (how many orders, what was charged for shipping) and
values that belong to the items (units and item value).

## Input tables

### `orders`

One row per order.

| Column | Type | Description |
|---|---|---|
| `order_id` | BIGINT | Unique per order. |
| `customer_id` | BIGINT | The customer who ordered. |
| `shipping_fee` | DECIMAL(10,2) | Charged once per order. Can be 0.00. |
| `items` | ARRAY&lt;STRUCT&lt;sku: STRING, qty: INT, unit_price: DECIMAL(10,2)&gt;&gt; | The line items. Can be an empty array, and can be NULL. |

Run `orders.printSchema()` to see the nested structure.

## Requirement

Return one row per customer.

- `orders` is the number of the customer's orders, whatever their items look like.
- `shipping_total` is the sum of `shipping_fee` over those orders. Each order's fee
  counts once.
- `items_value` is the sum of `qty` times `unit_price` over all items of all the
  customer's orders.
- `total_units` is the sum of `qty` over those items.
- An order with an empty or NULL `items` array still counts as an order and its
  shipping fee still counts. It adds nothing to `items_value` and `total_units`.
- A customer whose orders have no items at all has `items_value` = 0.00 and
  `total_units` = 0.

## Expected output

One row per customer.

| Column | Type | Description |
|---|---|---|
| `customer_id` | BIGINT | |
| `orders` | BIGINT | Number of orders. |
| `items_value` | DECIMAL(14,2) | Sum of qty times unit_price. |
| `shipping_total` | DECIMAL(12,2) | Sum of shipping fees, once per order. |
| `total_units` | BIGINT | Sum of qty. |

In [ ]:
orders.show(5)

## Your answer

Use the DataFrame API. The tables are available as DataFrames: `orders`.

In [ ]:
result = None  # Replace None with your DataFrame.

In [ ]:
check(result)